In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

mobility = pd.read_csv("data/mobility_clean.csv")
mobility['date'] = pd.to_datetime(mobility['date'])
mobility["year_month"] = mobility["date"].dt.to_period('M')

monthly = mobility.groupby(['sub_region_2', 'year_month']).mean().reset_index()

# mobility columns
mobility_feats = [
    'retail_and_recreation_percent_change_from_baseline', 
    'grocery_and_pharmacy_percent_change_from_baseline', 
    'transit_stations_percent_change_from_baseline', 
    'workplaces_percent_change_from_baseline', 
    'residential_percent_change_from_baseline'
]


rural_counties = ["Accomack County","Alleghany County","Bath County",
                  "Bland County","Brunswick County","Buchanan County",
                  "Carroll County","Charlotte County","Craig County",
                  "Dickenson County","Essex County","Grayson County",
                  "Greensville County","Halifax County","Henry County",
                  "Highland County","Lee County","Louisa County",
                  "Lunenburg County","Madison County","Mecklenburg County",
                  "Middlesex County","Montgomery County","Nelson County",
                  "Northampton County","Northumberland County","Patrick County",
                  "Pittsylvania County","Prince Edward County","Pulaski County",
                  "Richmond County","Rockbridge County","Rockingham County",
                  "Russell County","Smyth County","Southampton County",
                  "Tazewell County","Wise County","Wythe County","Shenandoah County"]

mobility['metro_label'] = np.where(mobility['sub_region_2'].isin(rural_counties), 0, 1)
monthly['metro_label'] = np.where(monthly['sub_region_2'].isin(rural_counties), 0, 1)

monthly.to_csv("data/monthly.csv")

# checkna
mobility_pca = mobility[mobility_feats]
monthly_pca = monthly[mobility_feats]

print(mobility_pca.isna().sum())
print(monthly_pca.isna().sum())


# justify assumptions

In [ ]:
short_names = {
    'retail_and_recreation_percent_change_from_baseline': 'Retail/Recreation',
    'grocery_and_pharmacy_percent_change_from_baseline': 'Grocery/Pharmacy',
    'transit_stations_percent_change_from_baseline': 'Transit Stations',
    'workplaces_percent_change_from_baseline': 'Workplaces',
    'residential_percent_change_from_baseline': 'Residential'
}

monthly = monthly.rename(columns=short_names)

mobility_feats_short = ['Retail/Recreation', 'Grocery/Pharmacy', 'Transit Stations', 'Workplaces', 'Residential']

In [ ]:
# linearity: scatterplots
import itertools

pairs = list(itertools.combinations(mobility_feats_short, 2))

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
axes = axes.flatten()

for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(data=monthly, x=x, y=y, ax=ax)
    ax.set_xlabel(x, style='italic')
    ax.set_ylabel(y, style='italic')
    ax.set_title(f"{x} vs {y}")

x_last, y_last = pairs[-1]

sns.scatterplot(data=monthly, x=x_last, y=y_last, ax=axes[10])
axes[10].set_title(f"{x_last} vs {y_last}")
axes[10].set_xlabel(x_last, style='italic')
axes[10].set_ylabel(y_last, style='italic')

# hide unused axes: 9 and 11
axes[9].set_visible(False)
axes[11].set_visible(False)

plt.tight_layout()
plt.show()



In [ ]:
print(monthly.columns)

In [ ]:
# scale data
scaler = StandardScaler()
mob_scaled = scaler.fit_transform(mobility_pca)

# pca
pca = PCA(n_components=2)
pca_result = pca.fit_transform(mob_scaled)

mobility['PC1'] = pca_result[:,0]
mobility['PC2'] = pca_result[:,1]

print(pca.explained_variance_ratio_)  # How much variance is captured


In [ ]:
# scale data
scaler = StandardScaler()
monthly_scaled = scaler.fit_transform(monthly_pca)

pca_m = PCA(n_components=2)
monthly_result = pca_m.fit_transform(monthly_scaled)

monthly['PC1'] = monthly_result[:,0]
monthly['PC2'] = monthly_result[:,1]
print(pca_m.explained_variance_ratio_)


In [ ]:
print(monthly.size)

In [ ]:
# visualize
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='metro_label', 
    data=mobility, 
    alpha=0.7,
    palette='tab20',
    legend=True
)
plt.xlabel("PC1")
plt.ylabel("PC2")

In [ ]:
# visualize
plt.figure(figsize=(10, 6))
sns.set_style(style="darkgrid")
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='metro_label', 
    data=monthly, 
    palette='tab20',
    legend=True
)
plt.title("Scatterplot of PC1 and PC2")

plt.xlabel("PC1")
plt.ylabel("PC2")

In [ ]:
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

plt.figure(figsize=(12,6))
for region, group in monthly.groupby('sub_region_2'):
    plt.plot(group['year_month_dt'], group['PC1'], alpha=0.5)
plt.xticks(rotation=45)
plt.title('PC1 over Time by Region')
plt.show()

In [ ]:
plt.figure(figsize=(12,6))

# Plot each region, coloring by metro status
for (region, metro_label), group in monthly.groupby(['sub_region_2','metro_label']):
    if metro_label == 1:  # Metro
        plt.plot(group['year_month_dt'], group['PC1'], color='lightblue', alpha=0.7)
    else:  # Non-metro
        plt.plot(group['year_month_dt'], group['PC1'], color='crimson', alpha=0.7)

plt.xticks(rotation=45)
plt.ylabel('PC1 (mobility pattern)')
plt.title('PC1 over Time by Region (Metro (gray) vs Non-Metro (red))')
plt.show()

In [ ]:
mob_feats_short = ["retail/rec", "grocery/pharma", "transit", "workplaces", "residential"]
loadings = pd.DataFrame(pca_m.components_.T, 
                        index=mob_feats_short,  # your original categories
                        columns=[f'PC{i+1}' for i in range(len(pca.components_))])


print(loadings['PC1'])

In [ ]:

short_names = {
    'retail_and_recreation_percent_change_from_baseline': 'Retail/Recreation',
    'grocery_and_pharmacy_percent_change_from_baseline': 'Grocery/Pharmacy',
    'transit_stations_percent_change_from_baseline': 'Transit Stations',
    'workplaces_percent_change_from_baseline': 'Workplaces',
    'residential_percent_change_from_baseline': 'Residential'
}
loadings_short = loadings.rename(index=short_names)

monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

# fig with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})

# top: pc1 over time
ax = axes[0]
sns.set_style(style="darkgrid")
sns.lineplot(
    data=monthly,
    x='year_month_dt',
    y='PC1',
    hue='metro_label',  # 1=Metro, 0=Rural
    units='sub_region_2',
    estimator=None,
    palette={0: 'crimson', 1: 'lightsteelblue'},
    alpha=0.7,
    ax=ax,
    legend='full'
)

ax.set_ylabel('PC1 (mobility pattern)')
ax.set_xlabel("Month")
ax.set_title('PC1 over Time by Region (Metro vs Non-Metro)')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Metro Label', labels=['Metro', 'Non-Metro'])

# --- Bottom subplot: PC1 loadings ---
ax2 = axes[1]
sns.set_style(style="darkgrid")
sns.barplot(
    x=loadings_short['PC1'].values,
    y=loadings_short['PC1'].index,
    hue=loadings_short['PC1'].index,
    palette='Set2',
    legend=False,
    ax=ax2
)
ax2.set_xlabel('Loading value')
ax2.set_title('PC1 Loadings by Mobility Category')

plt.tight_layout()
plt.show()

In [ ]:
# prep variance explained
expl_var = pca_m.explained_variance_ratio_
cum_var = np.cumsum(expl_var)

# Make a DataFrame for plotting
scree_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(expl_var))],
    'Variance': expl_var,
    'Cumulative': cum_var
})

plt.figure(figsize=(6,4))

# Barplot for individual variance
sns.set_style(style="darkgrid")
sns.barplot(x='PC', y='Variance', data=scree_df, color='teal', alpha=0.7)

# Lineplot for cumulative variance
sns.lineplot(x='PC', y='Cumulative', data=scree_df, marker='o', color='crimson')
plt.text(x=0.5, y=0.8, s="Cumulative Variance", color="crimson", fontsize=8)
plt.ylabel('Variance Explained')
plt.title('Scree Plot', fontsize=15)
plt.ylim(0,1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.text(-0.09, 0.7, s=f"{pca_m.explained_variance_ratio_[0]:.4f}")
plt.text(0.91, 0.15, s=f"{pca_m.explained_variance_ratio_[1]:.4f}")
plt.show()

In [ ]:

sns.set_style("ticks")
plt.figure(figsize=(10, 7))
cols = ["rosybrown", "lightsteelblue"]

sns.scatterplot(
    x='PC1', y='PC2', 
    hue='metro_label', palette=cols, 
    data=monthly, 
    alpha=0.7
)

for i in range(loadings.shape[0]): 
    plt.arrow(0, 0, loadings.PC1[i]*3, loadings.PC2[i]*3, 
              color='black', alpha=0.7, head_width=0.1)
    plt.text(loadings.PC1[i]*3.3, loadings.PC2[i]*3.3, loadings.index[i], 
             color='black', fontsize=12)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of County-Month Mobility Data\nArrows show feature contributions')
plt.legend(title='nonmetro = 0 / metro = 1')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.set_style(style="darkgrid")
sns.barplot(
    x=loadings['PC1'].values,
    y=loadings['PC1'].index,
    hue=loadings['PC1'].index,
    palette='Set2',
    legend=False
)
ax2.set_xlabel('Loading value')
ax2.set_title('PC1 Loadings by Mobility Category')
plt.ylabel("Mobility Category")
plt.title("PC1 Loadings", fontsize=15)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
monthly.to_csv("data/monthly.csv")